# S6E9: Smooth Keys & Triple TE

**Uploaded notebook baseline: 0.94632 public LB (Version 3).** This update trains from competition data only. Its new public score has not been measured.

The original notebook already had 45/39/16 logit weights, but lacked the smooth keys and third TE channel used in our later training experiments. This version adds those features to both LightGBM and CatBoost, while preserving the compact linear model.

| Change | Why |
|---|---|
| Income groups of $100 and $1,000; whole-km commute groups | Pool nearby values before frequency and target encoding |
| TE strengths auto, 10 and 100 for both main learners | Give sparse groups a more strongly regularized channel |
| Five outer folds by default | Match the previous tested training recipe; 15 fits instead of the uploaded notebook's 30 |
| One settings cell and consistent artifact export | Make the next run easier to inspect and reproduce |

Our separate later training recipe reached user-reported LB 0.94639. The public-prediction blend reached 0.94650; that score does **not** belong to this training notebook. No gain is guaranteed for this revision.

**Run:** attach S6E9 competition data, use CPU, and Run All. The final cell writes `/kaggle/working/submission.csv`. Use `FEATURE_PRESET = "reference"` and `FOLDS = 10` to restore the uploaded notebook's feature/fold configuration.


## Research provenance and design lineage

Public notebooks are treated here as research references rather than prediction sources. The table below credits work that materially influenced individual feature families, validation controls, or modelling ideas, while the implementation remains leak-aware and self-contained.

| Research idea | Public source(s) reviewed | How it appears here |
|---|---|---|
| Fixed comparisons, noise controls, histogram resolution | Georgy Mamarin [3] | Representation changes are separated from model-capacity changes, with controlled baselines preserved. |
| Shallow LightGBM, digit features, dual target encoding | Naji [4] | Explicit digit representations and cross-fitted target-mean channels are retained. |
| LightGBM representation experiments | Kirill [5] | Numeric representation is treated as a first-class experimental variable. |
| Nested target/frequency encoding | Mizushima [6] | Supervised encodings are fitted inside training partitions, with validation/test labels excluded from feature fitting. |
| Multiscale income neighborhoods | jazivxt [7] | Central, neighbor, slope, curvature, and count channels are built on stable income bins. |
| Income modulo diagnostics | megayak [8] | Modulo features remain an optional ablation rather than an assumed improvement. |
| Compact linear-leaf model and Gaussian income structure | Sergey [9] | A compact linear-tree member and a leak-safe integer-grid Gaussian neighborhood channel are included. |
| Charging/policy interactions | Koushik [10] | Domain interaction formulas remain isolated as a separate feature switch. |
| Multiple target-encoding smoothing strengths | Rugved [11] | Multiple smoothing channels share the same cross-fitting scheme. |
| Original-data analysis and reconstruction | Chris Deotte [12], Yusuke Hayashi [13] | Original-data mappings remain optional and disabled by default; reconstruction is not treated as proof of the competition generator. |

API behavior and parameter semantics are grounded in the LightGBM parameter documentation [14], LightGBM tuning documentation [15], and scikit-learn `TargetEncoder` documentation [2].


## 1. Imports and configuration schemas

Feature choices and learner choices live in separate dataclasses. `FeatureConfig` owns representation, smoothing, frequency scope, neighborhood statistics, and leakage controls. `ModelConfig` owns the learner, objective, capacity, regularization, thread count, and training budget.

The default LightGBM path stays on CPU, while XGBoost and CatBoost are imported only when selected. Keeping the configuration explicit also makes each saved artifact traceable to the experiment that created it.


In [ ]:
"""S6E9: configurable feature-engineering and modelling pipelines.

Research provenance and usage are explained in the accompanying Kaggle notebook.
Only competition CSVs are required. No public predictions are downloaded.
"""
from __future__ import annotations

import copy
import gc
import hashlib
import json
import os
import time
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.ndimage import convolve1d
from scipy.special import expit, logit
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from sklearn.preprocessing import TargetEncoder

TARGET, ID = 'Will_Buy_EV', 'id'
NUMERIC = ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned',
           'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
CATEGORIES = {
    'Gender': ['Male', 'Female', 'Other'],
    'City_Type': ['Urban', 'Suburban', 'Rural'],
    'Current_Car_Type': ['Sedan', 'SUV', 'Hatchback', 'Truck'],
    'Home_Charging_Possible': ['No', 'Yes'], 'Subsidy_Available': ['No', 'Yes'],
    'Range_Anxiety_Level': ['Low', 'Medium', 'High'],
}
RAW = NUMERIC + list(CATEGORIES)
IMPLEMENTATION_VERSION = '2026-09-14.smooth-te100'



@dataclass
class FeatureConfig:
    """Change features independently from the learner and its parameters."""
    smooth_keys: bool = False
    digits: bool = True
    floating_digit_artifacts: bool = False  # deliberate reproduction of public float-floor features
    income_moduli: tuple[int, ...] = ()
    frequency: bool = True
    frequency_scope: str = 'fold'  # 'fold' or explicit target-free 'train_test'
    target_encoding: bool = True
    te_smoothing: tuple = ('auto', 10.0)
    te_columns: tuple[str, ...] = ()  # empty = raw columns + digit/modulo keys
    inner_folds: int = 5
    inner_seed: int = 17
    income_bins: tuple[int, ...] = ()
    neighborhood_smoothing: float = 10.0
    neighborhood_channels: tuple[str, ...] = ('central', 'neighbor', 'slope', 'curvature', 'count')
    income_kernel_sigma: float = 0.0  # compact integer-grid kernel; 0 disables
    interactions: bool = False
    recipe: bool = False
    boundary_flags: bool = False
    native_categories: bool = False
    scale_numeric: bool = False
    original_path: str = ''  # optional public original-data CSV, disabled by default
    compact: bool = False  # compact representation for linear-leaf experiments


@dataclass
class ModelConfig:
    """All estimator settings live in this second configuration."""
    learner: str = 'lightgbm'  # lightgbm, xgboost, catboost
    rounds: int = 50000
    patience: int = 500
    threads: int = 4
    seed: int = 42
    params: dict = field(default_factory=lambda: dict(
        objective='binary', metric='auc', learning_rate=.02,
        max_depth=5, num_leaves=31, min_data_in_leaf=10,
        feature_fraction=.30, bagging_fraction=1.0, bagging_freq=0,
        lambda_l1=.071, lambda_l2=2.0, max_bin=1024,
        feature_pre_filter=False, device_type='cpu', verbosity=-1,
    ))


@dataclass
class Experiment:
    name: str
    features: FeatureConfig = field(default_factory=FeatureConfig)
    model: ModelConfig = field(default_factory=ModelConfig)


## 2. Settings: the improved training recipe

Change the settings at the top of the next cell. `smooth_keys_te100` is the default; `reference` preserves the uploaded features. Model parameters and rounds stay fixed so this is primarily a feature update.

| Member | Weight | Rounds | Features |
|---|---:|---:|---|
| LightGBM | 0.45 | 1,377 | Digits, smooth keys, triple TE, income neighborhoods |
| CatBoost | 0.39 | 944 | Digits, smooth keys, triple TE, income neighborhoods |
| Linear LightGBM | 0.16 | 967 | Original compact representation and Gaussian income kernel |

The default uses five shared outer folds. Ten folds remain optional, but are a different experiment and cost twice as many fits. Outer-fold early stopping stays off. Historical feature and parameter selection means OOF scores remain development estimates.


In [ ]:
DATA_DIR = None  # Kaggle paths are detected; locally set this to "dataset".
FEATURE_PRESET = "smooth_keys_te100"  # or "reference"
RUN_NAME = "s6e9_smooth_te100_v2"  # change when changing folds or settings
RUN_MODE = "submission"  # "submission" or "research"
SMOKE_TEST = os.environ.get("S6E9_SMOKE_TEST", "0") == "1"
FOLDS = 5  # optional: 10; default matches the earlier tested recipe
FOLD_SEED = 42
MODEL_SEEDS = (42,)
THREADS = 4
EXTRA_TUNING_TRIALS = 0  # Research only; try 12, then assess a few finalists.
EARLY_STOP_ON_OUTER_FOLD = False  # True makes reported OOF more optimistic.

FROZEN_RECIPE = {'experiments': [{'name': 'smooth1_bagging',
                  'features': {'income_bins': [8192, 16384], 'neighborhood_smoothing': 1.0},
                  'model': {'learner': 'lightgbm',
                            'rounds': 1377,
                            'patience': 250,
                            'threads': 4,
                            'seed': 42,
                            'params': {'objective': 'binary',
                                       'metric': 'auc',
                                       'learning_rate': 0.02,
                                       'max_depth': 5,
                                       'num_leaves': 31,
                                       'min_data_in_leaf': 10,
                                       'feature_fraction': 0.3,
                                       'bagging_fraction': 0.85,
                                       'bagging_freq': 1,
                                       'lambda_l1': 0.071,
                                       'lambda_l2': 2.0,
                                       'max_bin': 1024,
                                       'feature_pre_filter': False,
                                       'device_type': 'cpu',
                                       'verbosity': -1}}},
                 {'name': 'catboost_neighborhood',
                  'features': {'income_bins': [8192, 16384]},
                  'model': {'learner': 'catboost',
                            'rounds': 944,
                            'patience': 200,
                            'threads': 4,
                            'seed': 42,
                            'params': {'loss_function': 'Logloss',
                                       'eval_metric': 'AUC',
                                       'depth': 5,
                                       'learning_rate': 0.04,
                                       'l2_leaf_reg': 5.0,
                                       'rsm': 0.5}}},
                 {'name': 'linear_nested',
                  'features': {'digits': False,
                               'te_smoothing': [0.5],
                               'te_columns': ['Age',
                                              'Gender',
                                              'Number_of_Cars_Owned',
                                              'City_Type',
                                              'Current_Car_Type',
                                              'Range_Anxiety_Level'],
                               'neighborhood_smoothing': 0.5,
                               'income_kernel_sigma': 5.0,
                               'scale_numeric': True,
                               'compact': True},
                  'model': {'learner': 'lightgbm',
                            'rounds': 967,
                            'patience': 180,
                            'threads': 4,
                            'seed': 42,
                            'params': {'objective': 'binary',
                                       'metric': 'auc',
                                       'learning_rate': 0.02,
                                       'max_depth': -1,
                                       'num_leaves': 31,
                                       'min_data_in_leaf': 10,
                                       'feature_fraction': 0.3,
                                       'bagging_fraction': 0.9,
                                       'bagging_freq': 1,
                                       'lambda_l1': 0.05,
                                       'lambda_l2': 10,
                                       'max_bin': 1023,
                                       'feature_pre_filter': False,
                                       'device_type': 'cpu',
                                       'verbosity': -1,
                                       'linear_tree': True,
                                       'linear_lambda': 0.5}}}],
 'weights': [0.45, 0.39, 0.16],
 'kind': 'logit'}

# Apply the selected features to both main learners before building configs.
if FEATURE_PRESET not in ("reference", "smooth_keys_te100"):
    raise ValueError("Choose reference or smooth_keys_te100")
if FEATURE_PRESET == "smooth_keys_te100":
    for item in FROZEN_RECIPE["experiments"][:2]:
        item["features"].update(smooth_keys=True, te_smoothing=["auto", 10.0, 100.0])

# Feature engineering pipeline configuration — change these independently.
FEATURE_CONFIGS = {
    item["name"]: FeatureConfig(**item["features"])
    for item in FROZEN_RECIPE["experiments"]
}
# Modelling pipeline configuration — each learner has a separate parameter dict.
MODEL_CONFIGS = {
    item["name"]: ModelConfig(**dict(item["model"], threads=THREADS))
    for item in FROZEN_RECIPE["experiments"]
}
ACTIVE_EXPERIMENTS = list(FEATURE_CONFIGS)
BLEND_WEIGHTS = FROZEN_RECIPE["weights"]
BLEND_KIND = FROZEN_RECIPE["kind"]

# Example next feature experiment (replace the name with a member above):
# name = ACTIVE_EXPERIMENTS[0]
# FEATURE_CONFIGS[name] = replace(FEATURE_CONFIGS[name], income_moduli=(100, 1000))
# MODEL_CONFIGS[name].params["max_bin"] = 4095

EXPERIMENTS = [Experiment(name, FEATURE_CONFIGS[name], MODEL_CONFIGS[name])
               for name in ACTIVE_EXPERIMENTS]
for exp in EXPERIMENTS:
    print(exp.name, "|", exp.model.learner, "|", exp.model.rounds, "rounds")
print("Frozen weights:", dict(zip(ACTIVE_EXPERIMENTS, BLEND_WEIGHTS)))


### Why these three members coexist

The main LightGBM member is the strongest selected single-model anchor. CatBoost adds a different tree-building bias over nearly the same engineered signal, while the linear-leaf LightGBM contributes a compact representation and linear responses that create useful prediction diversity even with a slightly lower standalone selection AUC.

For logit averaging, each probability is transformed with `log(p / (1-p))`, combined with the frozen weights, and mapped back through the inverse transform. Extreme probabilities are clipped first so the calculation remains finite.

Inactive feature families remain in the configuration schema for provenance and ablation history, while the active recipe above defines the default submission workflow.


## 3. Data integrity and schema checks

`Yes` is mapped to the positive class, while `id` is excluded from the feature matrix. Before modelling, the notebook checks the expected schema, target mapping, unique IDs, train/test ID separation, and sample-submission alignment.

The displayed schema summary is descriptive only; its full-data target summaries are never reused as model features.


In [ ]:
def file_hash(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def load_competition(data_dir=None):
    candidates = ([Path(data_dir)] if data_dir else []) + [
        Path('/kaggle/input/competitions/playground-series-s6e9'),
        Path('/kaggle/input/playground-series-s6e9'), Path('dataset')]
    root = next((p for p in candidates if all((p/f).exists() for f in
                 ['train.csv', 'test.csv', 'sample_submission.csv'])), None)
    if root is None:
        raise FileNotFoundError('Attach the S6E9 competition data or set DATA_DIR.')
    train, test, sample = [pd.read_csv(root/f) for f in ['train.csv','test.csv','sample_submission.csv']]
    if set(train.columns) != set(RAW+[ID,TARGET]) or set(test.columns) != set(RAW+[ID]):
        raise ValueError('Unexpected competition schema')
    y = train[TARGET].map({'No':0, 'Yes':1})
    if y.isna().any() or set(y.unique()) != {0,1}:
        raise ValueError('Expected Yes/No target')
    for frame in [train,test,sample]:
        if frame[ID].isna().any() or not frame[ID].is_unique:
            raise ValueError('IDs must be present and unique')
    if set(sample.columns) != {ID,TARGET} or set(test[ID]) != set(sample[ID]):
        raise ValueError('Submission schema/IDs differ from test')
    if not set(train[ID]).isdisjoint(set(test[ID])):
        raise ValueError('Training and test IDs overlap')
    return train, test, sample, y.to_numpy('int8'), root

train, test, sample_submission, y, data_root = load_competition(DATA_DIR)
print(f"Train: {len(train):,} | test: {len(test):,} | positive rate: {y.mean():.4%}")
print("Data:", data_root)
schema = pd.DataFrame({"dtype": train[RAW].dtypes.astype(str),
                       "unique_train": train[RAW].nunique(),
                       "missing_train": train[RAW].isna().sum()})
display(schema)
if SMOKE_TEST:
    train = train.sample(min(1800, len(train)), random_state=917).reset_index(drop=True)
    test = test.head(800).copy()
    sample_submission = sample_submission[sample_submission[ID].isin(test[ID])].copy()
    y = train[TARGET].map({"No": 0, "Yes": 1}).to_numpy("int8")
    FOLDS, MODEL_SEEDS = 2, (42,)
    for exp in EXPERIMENTS:
        exp.features.inner_folds = 3
        exp.model.rounds, exp.model.patience = 20, 5
    print("SMOKE TEST: reduced rows and iterations; do not submit these predictions.")


## 4. Feature engineering: exact values without target leakage

For a categorical or exact-value key, the notebook uses a smoothed target mean

\[
TE(v)=\frac{\sum_{i:x_i=v} y_i + \alpha\,\bar y}{n_v+\alpha}.
\]

These supervised features are cross-fitted on inner folds. A training row receives an encoding learned from other rows, while an outer validation or test row receives the mapping fitted on the complete outer-training partition. Unseen values fall back to the fitted prior. This follows the role of smoothed target encoding described by scikit-learn [2], while the nested fitting structure keeps outer labels out of feature construction.

For income neighborhoods, target-free bin edges are fitted once on the outer-training features and reused across inner folds. Target sums, counts, central means, neighbor means, slope, curvature, and count channels are then learned only from the permitted training labels. The optional Gaussian channel uses an integer grid rather than a dense pairwise matrix.

### Representation families in the pipeline

| Family | Representation | Leakage status |
|---|---|---|
| Raw numeric + ordinal categories | Original competition features | Target-free |
| Income digits / commute digits | Exact low-order numeric structure | Target-free |
| Frequencies | Exact-value prevalence | Target-free within the configured population |
| Target encoding | Smoothed conditional target means | Cross-fitted |
| Income neighborhoods | Local target structure across stable numeric bins | Cross-fitted |
| Gaussian income kernel | Smooth local income target structure | Cross-fitted |
| Domain interactions | Charging, subsidy, commute, anxiety relations | Target-free |
| Compact scaled matrix | Small representation for linear leaves | Scaling uses target-free columns only |

The central leakage rule is simple: validation and test labels never enter feature fitting.

### New smooth keys and triple encoding

Income 92,887 maps to keys 928 and 92; commute 23.4 maps to 23. Each key gets a fold-local frequency and three cross-fitted target encodings. Every inner held-out row is excluded from the rates and prior used to encode it. The compact linear member keeps its original settings.

These additions follow the [public coarse-key/TE research](https://www.kaggle.com/code/najiama/pure-lgbm-model-cv-0-94606-lb-0-94637) and our earlier controlled experiments. Recent expanded ladders and window variants failed to establish an audit gain over the stronger blend, so they are not enabled here.


In [ ]:
class FeaturePipeline:
    """Fit target-free state once, cross-fit supervised training representations.

    Public API: fit_transform(X, y, transductive_pool=None), transform(X).
    Validation/test labels are never accepted. The returned training matrix uses
    inner-held-out target encodings; transform uses all fitted training labels.
    """
    def __init__(self, config: FeatureConfig):
        self.config = copy.deepcopy(config)

    def _base(self, X):
        c = self.config
        f = X[RAW].copy().reset_index(drop=True)
        for col in NUMERIC:
            f[col] = pd.to_numeric(f[col], errors='coerce').fillna(self.medians_[col])
        for col, levels in CATEGORIES.items():
            f[col] = f[col].map({v:i for i,v in enumerate(levels)}).fillna(-1).astype('int16')
        income = np.rint(f.Annual_Income_USD).astype('int64')
        commute = np.rint(10*f.Daily_Commute_km).astype('int64')
        if c.digits:
            for power in range(5):
                f[f'income_digit_{power}'] = (income // 10**power) % 10
            f['commute_tenths'] = commute % 10
            f['commute_units'] = (commute // 10) % 10
        if c.floating_digit_artifacts:
            for col in NUMERIC:
                for power in range(-4,4):
                    f[f'float_digit__{col}_{power}'] = np.floor_divide(f[col],10.0**power)%10
        for modulus in c.income_moduli:
            f[f'income_mod_{modulus}'] = income % modulus
        home, subsidy = f.Home_Charging_Possible, f.Subsidy_Available
        if c.interactions or c.compact:
            f['homecharge'] = f.Charging_Stations_Near_Home + 5*home
        if c.interactions:
            total = f.Charging_Stations_Near_Home + f.Charging_Stations_Near_Work
            f['chargers_total'] = total
            f['commute_per_charger'] = f.Daily_Commute_km/(1+total)
            f['home_subsidy'] = home*subsidy
            f['subsidy_income'] = subsidy*f.Annual_Income_USD/100000
            f['subsidy_concern'] = subsidy*f.Environmental_Concern_Level
        if c.recipe:
            penalty = f.Range_Anxiety_Level.map({0:0,1:1,2:3}).fillna(0)
            f['purchase_recipe'] = 1.2*f.Annual_Income_USD/100000 + .6*f.Environmental_Concern_Level + 2*subsidy-penalty-5.5
            f['anxiety_recipe'] = f.Daily_Commute_km - 5*(f.Charging_Stations_Near_Home+f.Charging_Stations_Near_Work)-150*home
        if c.boundary_flags:
            f['income_floor'] = (income==30000).astype('int8')
            f['income_high'] = (income>=170537).astype('int8')
            f['income_low_region'] = income.between(31004,41970).astype('int8')
            f['concern_floor'] = (f.Environmental_Concern_Level==1).astype('int8')
        if c.smooth_keys:
            # Pool nearby values; all target rates are still nested below.
            f['smooth_key_income100'] = income // 100
            f['smooth_key_income1000'] = income // 1000
            f['smooth_key_commute_integer'] = commute // 10
        return f

    def _fit_supervised(self, base, y):
        c = self.config
        state = {'prior':float(np.mean(y)), 'encoders':[], 'bins':[]}
        if c.target_encoding:
            keys = base[self.te_columns_].astype(str)
            for smoothing in c.te_smoothing:
                enc = TargetEncoder(smooth=smoothing, target_type='binary')
                enc.fit(keys, y)  # Only this fit subset contributes, including its prior.
                state['encoders'].append(enc)
        income = base.Annual_Income_USD.to_numpy(float)
        for bins in c.income_bins:
            # Target-free edges are fitted on outer-training features and shared
            # by the inner folds, so a channel keeps the same meaning in each fit.
            edges = self.income_edges_[bins]
            codes = np.searchsorted(edges[1:-1],income)
            sums = np.bincount(codes,weights=y,minlength=bins)
            counts = np.bincount(codes,minlength=bins).astype(float)
            smooth = c.neighborhood_smoothing
            central = (sums+smooth*state['prior'])/(counts+smooth)
            weights = np.exp(-.5*(np.arange(-2,3)/.8)**2)
            weighted_sum = convolve1d(sums,weights,mode='constant',cval=0)
            weighted_count = convolve1d(counts,weights,mode='constant',cval=0)
            neighbor = (weighted_sum+smooth*state['prior'])/(weighted_count+smooth)
            left = np.r_[state['prior'],central[:-1]]
            right = np.r_[central[1:],state['prior']]
            values = dict(central=central, neighbor=neighbor, left=left, right=right,
                          slope=right-left, curvature=central-.5*(left+right), count=np.log1p(counts))
            state['bins'].append((edges, values))
        if c.income_kernel_sigma>0:
            values = np.rint(income).astype('int64')
            lo, hi = int(values.min()), int(values.max())
            if hi-lo>2_000_000:
                raise ValueError('Unexpected income range for grid kernel')
            counts = np.bincount(values-lo,minlength=hi-lo+1).astype(float)
            sums = np.bincount(values-lo,weights=y,minlength=hi-lo+1)
            radius = max(1,int(np.ceil(8*c.income_kernel_sigma)))
            kernel = np.exp(-.5*(np.arange(-radius,radius+1)/c.income_kernel_sigma)**2)
            num = convolve1d(sums,kernel,mode='constant',cval=0)
            den = convolve1d(counts,kernel,mode='constant',cval=0)
            state['kernel'] = (lo,(num+c.neighborhood_smoothing*state['prior'])/(den+c.neighborhood_smoothing))
        return state

    def _supervised(self, base, state):
        blocks, names = [], []
        if state['encoders']:
            keys=base[self.te_columns_].astype(str)
            for i,enc in enumerate(state['encoders']):
                blocks.append(enc.transform(keys))
                names += [f'te{i}__{col}' for col in self.te_columns_]
        income=base.Annual_Income_USD.to_numpy(float)
        for q,(edges,values) in zip(self.config.income_bins,state['bins']):
            codes=np.searchsorted(edges[1:-1],income)
            for channel in self.config.neighborhood_channels:
                blocks.append(values[channel][codes,None])
                names.append(f'income_bin{q}__{channel}')
        if 'kernel' in state:
            lo, lookup=state['kernel']; codes=np.rint(income).astype('int64')-lo
            valid=(codes>=0)&(codes<len(lookup))
            result=np.full(len(base),state['prior']); result[valid]=lookup[codes[valid]]
            blocks.append(result[:,None]); names.append('income_kernel')
        array=np.column_stack(blocks).astype('float32') if blocks else np.empty((len(base),0),dtype='float32')
        return pd.DataFrame(array,columns=names)

    def _unsupervised(self, base):
        c=self.config
        columns = (['Annual_Income_USD','Daily_Commute_km','Charging_Stations_Near_Work',
                    'Environmental_Concern_Level','Home_Charging_Possible','Subsidy_Available',
                    'Range_Anxiety_Level','homecharge'] if c.compact else list(base.columns))
        out=base[columns].copy().astype('float32')
        if c.scale_numeric:
            # Scale only target-free features. Scaling TE on all training encodings
            # would reintroduce a tiny self-label dependence through its mean/std.
            out=(out-self.scale_mean_[columns])/self.scale_std_[columns]
        if c.frequency:
            for col,mapping in self.frequencies_.items():
                out[f'freq__{col}']=base[col].map(mapping).fillna(0).astype('float32')
        if self.original_maps_:
            for col,mapping in self.original_maps_.items():
                out[f'original__{col}']=base[col].astype(str).map(mapping).fillna(self.original_prior_).astype('float32')
        return out

    def fit_transform(self, X, y, transductive_pool=None):
        c=self.config
        if len(X)!=len(y) or len(X)<c.inner_folds or set(np.unique(y))!={0,1}:
            raise ValueError('Feature fit needs matching rows, both classes and enough inner-fold rows')
        if c.frequency_scope not in ('fold','train_test') or c.inner_folds<2:
            raise ValueError('Invalid frequency scope or inner fold count')
        if any(m<2 for m in c.income_moduli) or any(q<2 for q in c.income_bins) or c.neighborhood_smoothing<=0:
            raise ValueError('Invalid feature hyperparameters')
        self.medians_=X[NUMERIC].median().fillna(0)
        base=self._base(X)
        income=base.Annual_Income_USD.to_numpy(float)
        lo,hi=float(income.min()),float(income.max())
        self.income_edges_={q:np.linspace(lo,max(lo+1,hi),q+1) for q in c.income_bins}
        self.te_columns_=list(c.te_columns) if c.te_columns else [col for col in base if col in RAW or 'digit' in col or 'commute_' in col or 'income_mod_' in col or 'smooth_key_' in col]
        if not set(self.te_columns_)<=set(base):
            raise ValueError('TE column missing from generated features')
        self.scale_mean_=base.mean()
        self.scale_std_=base.std(ddof=0).replace(0,1).fillna(1)
        pool=base
        if c.frequency_scope=='train_test':
            if transductive_pool is None:
                raise ValueError('train_test frequency requires an explicitly supplied feature-only pool')
            pool=self._base(transductive_pool)
        self.frequencies_={col:pool[col].value_counts(normalize=True) for col in self.te_columns_} if c.frequency else {}
        self.original_maps_={}
        if c.original_path:
            original=pd.read_csv(c.original_path)
            if not set(RAW+[TARGET])<=set(original):
                raise ValueError('Original-data CSV has unexpected columns')
            labels=original[TARGET].map({'Yes':1,'No':0})
            original=original.loc[labels.notna()].copy(); labels=labels[labels.notna()].to_numpy()
            raw=self._base(original)
            self.original_prior_=float(labels.mean())
            for col in RAW:
                self.original_maps_[col]=pd.DataFrame({'k':raw[col].astype(str),'y':labels}).groupby('k').y.mean()
        self.supervised_state_=self._fit_supervised(base,np.asarray(y))
        template=self._supervised(base.iloc[:1],self.supervised_state_)
        encoded=np.empty((len(base),template.shape[1]),dtype='float32')
        if template.shape[1]:
            for fit,hold in KFold(c.inner_folds,shuffle=True,random_state=c.inner_seed).split(base):
                # Refit supervised state and priors; keep target-free edges shared.
                state=self._fit_supervised(base.iloc[fit],np.asarray(y)[fit])
                encoded[hold]=self._supervised(base.iloc[hold],state).to_numpy()
        result=pd.concat([self._unsupervised(base),pd.DataFrame(encoded,columns=template.columns)],axis=1)
        # Only drop constants from target-free columns. Supervised feature pruning
        # on all labels would itself be a supervised selection step.
        keep=[col for col in result if col in template.columns or result[col].nunique(dropna=False)>1]
        self.columns_=keep
        return self._finish(result)

    def _finish(self, result):
        result=result[self.columns_].copy()
        if not np.isfinite(result.to_numpy(dtype=float)).all():
            raise ValueError('Feature pipeline produced nonfinite values')
        if self.config.native_categories and not self.config.scale_numeric:
            for col,levels in CATEGORIES.items():
                if col in result:
                    result[col]=pd.Categorical(result[col],categories=list(range(len(levels))))
        return result

    def transform(self, X):
        if not hasattr(self,'columns_'):
            raise RuntimeError('Fit FeaturePipeline before transform')
        base=self._base(X)
        return self._finish(pd.concat([self._unsupervised(base),self._supervised(base,self.supervised_state_)],axis=1))


## 5. Modelling pipeline

LightGBM remains the primary learner because shallow trees can exploit the engineered exact-value and neighborhood channels without relying on extreme depth. CatBoost and XGBoost share the same modelling interface, while the frozen recipe uses CatBoost as its complementary non-LightGBM member.

The wrapper also guards against two easy-to-miss LightGBM configuration problems: requesting row sampling with `bagging_freq=0`, and setting `num_leaves` above the capacity implied by the selected depth. Learner-specific parameters stay explicit, and prediction column order is preserved between fitting and inference.

Each fitted member can save its model, matching feature pipeline, importance table, configuration context, validation predictions, and test predictions.


In [ ]:
class ModellingPipeline:
    """Fit one estimator; prediction, model persistence and importance are uniform."""
    def __init__(self, config: ModelConfig):
        self.config=copy.deepcopy(config)

    def fit(self, X, y, X_stop=None, y_stop=None):
        c=self.config
        if c.rounds<1 or c.threads<1 or c.patience<1:
            raise ValueError('rounds, patience and threads must be positive')
        start=time.perf_counter()
        self.feature_names_=list(X.columns)
        if c.learner=='lightgbm':
            params=dict(c.params,seed=c.seed,num_threads=c.threads)
            if params.get('max_depth',-1)>0 and params.get('num_leaves',31)>2**params['max_depth']:
                raise ValueError('num_leaves exceeds the selected depth capacity')
            if params.get('bagging_fraction',1)<1 and params.get('bagging_freq',0)==0:
                raise ValueError('Set bagging_freq>0 to activate requested row bagging')
            train=lgb.Dataset(X,label=y)
            valid=[] if X_stop is None else [lgb.Dataset(X_stop,label=y_stop,reference=train)]
            callbacks=[lgb.log_evaluation(0)]
            if valid:
                callbacks.append(lgb.early_stopping(c.patience,verbose=False))
            self.model_=lgb.train(params,train,num_boost_round=c.rounds,valid_sets=valid,callbacks=callbacks)
            self.best_iteration_=self.model_.best_iteration or c.rounds
        elif c.learner=='xgboost':
            from xgboost import XGBClassifier
            params=dict(c.params,n_estimators=c.rounds,n_jobs=c.threads,random_state=c.seed)
            if X_stop is not None:
                params['early_stopping_rounds']=c.patience
            self.model_=XGBClassifier(**params)
            self.model_.fit(X,y,eval_set=None if X_stop is None else [(X_stop,y_stop)],verbose=False)
            self.best_iteration_=c.rounds if X_stop is None else self.model_.best_iteration+1
        elif c.learner=='catboost':
            from catboost import CatBoostClassifier
            params=dict(c.params,iterations=c.rounds,thread_count=c.threads,random_seed=c.seed,
                        verbose=False,allow_writing_files=False)
            self.model_=CatBoostClassifier(**params)
            self.model_.fit(X,y,eval_set=None if X_stop is None else (X_stop,y_stop),
                            early_stopping_rounds=None if X_stop is None else c.patience)
            self.best_iteration_=self.model_.tree_count_
        else:
            raise ValueError(f'Unknown learner: {c.learner}')
        self.seconds_=time.perf_counter()-start
        return self

    def predict(self, X):
        if list(X.columns)!=self.feature_names_:
            raise ValueError('Prediction schema/order differs from model fit')
        if self.config.learner=='lightgbm':
            p=self.model_.predict(X,num_iteration=self.best_iteration_,num_threads=self.config.threads)
        else:
            p=self.model_.predict_proba(X)[:,1]
        p=np.asarray(p,dtype=float)
        if not np.isfinite(p).all() or ((p<0)|(p>1)).any():
            raise ValueError('Invalid probability predictions')
        return p

    def save(self, stem):
        path=Path(stem)
        suffix={'lightgbm':'.txt','xgboost':'.ubj','catboost':'.cbm'}[self.config.learner]
        self.model_.save_model(str(path.with_suffix(suffix)))
        return path.with_suffix(suffix)

    def importance(self):
        if self.config.learner=='lightgbm':
            values=self.model_.feature_importance(importance_type='gain')
        else:
            values=self.model_.feature_importances_
        return pd.DataFrame({'feature':self.feature_names_,'importance':values}).sort_values('importance',ascending=False)


def make_experiment(name, base=None, features=None, params=None, **model_changes):
    exp=copy.deepcopy(base) if base else Experiment(name)
    exp.name=name
    if features:
        exp.features=replace(exp.features,**features)
    if params:
        exp.model.params.update(params)
    if model_changes:
        exp.model=replace(exp.model,**model_changes)
    return exp


def blend_predictions(predictions, weights, kind='logit'):
    weights=np.asarray(weights,dtype=float)
    if len(weights)!=predictions.shape[1] or (weights<0).any() or not np.isclose(weights.sum(),1):
        raise ValueError('Blend weights must be nonnegative and sum to one')
    if kind=='probability':
        return predictions@weights
    if kind=='logit':
        return expit(logit(np.clip(predictions,1e-7,1-1e-7))@weights)
    raise ValueError('Use probability or logit blending')


def save_submission(prediction, test_ids, sample, path):
    prediction=np.asarray(prediction)
    if len(prediction)!=len(test_ids) or not np.isfinite(prediction).all() or ((prediction<0)|(prediction>1)).any():
        raise ValueError('Invalid submission probabilities')
    if not pd.Index(test_ids).is_unique or set(test_ids)!=set(sample[ID]):
        raise ValueError('Test IDs do not match submission IDs')
    submission=sample[[ID]].copy()
    submission[TARGET]=submission[ID].map(pd.Series(prediction,index=test_ids))
    if submission[TARGET].isna().any():
        raise ValueError('Unmapped submission IDs')
    submission.to_csv(path,index=False)


## 6. Validation and reproducibility

Controlled local research used a **70% fit / 10% stopping / 10% selection / 10% audit** development split. Candidate features and blend weights were chosen on the selection partition, then the frozen recipe was evaluated on the audit partition.

That audit is not presented as perfectly untouched real-world data because the same overall training population had already influenced earlier project work. Uncertainty is therefore reported explicitly, and local AUC is kept separate from public leaderboard AUC.

Production cross-validation uses shared stratified folds across all members. With fixed iteration counts, outer validation labels do not choose stopping rounds. Test predictions are averaged across folds and model seeds before the frozen blend is applied.

Reproducibility checks fingerprint the train/test/sample data, experiment configuration, package versions, fold settings, seeds, blend settings, and implementation revision. Completed-fold markers are written last, so interrupted partial fits are not silently accepted as complete runs.


In [ ]:
def paired_auc_interval(y, baseline, candidate, seed=42, repeats=300):
    """Stratified paired bootstrap with pre-sorted predictions and exact tie handling."""
    rng=np.random.default_rng(seed)
    classes=[np.flatnonzero(y==k) for k in [0,1]]
    ordering=[]
    for prediction in [baseline,candidate]:
        order=np.argsort(prediction,kind='stable')
        starts=np.r_[0,np.flatnonzero(np.diff(prediction[order]))+1]
        ordering.append((order,starts))
    diffs=[]
    for _ in range(repeats):
        sample=np.concatenate([rng.choice(idx,len(idx),replace=True) for idx in classes])
        weight=np.bincount(sample,minlength=len(y)); scores=[]
        for order,starts in ordering:
            positive=np.add.reduceat(weight[order]*y[order],starts)
            negative=np.add.reduceat(weight[order]*(1-y[order]),starts)
            scores.append(np.sum(positive*(np.cumsum(negative)-.5*negative))/(positive.sum()*negative.sum()))
        diffs.append(scores[1]-scores[0])
    return np.quantile(diffs,[.025,.975]).tolist()


def run_cross_validation(experiments, train, test, sample, weights, output_dir,
                         folds=5, fold_seed=42, model_seeds=(42,), blend_kind='logit',
                         early_stopping=False, resume=True):
    """Train configurable members on shared folds and export component/final CSVs.

    Default fixed iteration counts must be chosen before this run. Setting
    early_stopping=True uses the outer fold for stopping and makes OOF more
    optimistic. Any earlier feature/model selection also limits OOF independence.
    """
    import joblib
    import re
    from importlib.metadata import version
    if len({e.name for e in experiments})!=len(experiments) or not experiments:
        raise ValueError('Experiment names must be unique and nonempty')
    if any(not re.fullmatch(r'[A-Za-z0-9_-]+',e.name) for e in experiments):
        raise ValueError('Use simple experiment names containing letters, numbers and underscores')
    if folds<2 or not model_seeds:
        raise ValueError('At least two folds and one model seed are required')
    blend_predictions(np.full((2,len(experiments)),.5),weights,blend_kind)
    out=Path(output_dir); out.mkdir(parents=True,exist_ok=True)
    y=train[TARGET].map({'No':0,'Yes':1}).to_numpy('int8')
    X,XT=train[RAW],test[RAW]
    fingerprint=lambda f: hashlib.sha256(pd.util.hash_pandas_object(f,index=True).values.tobytes()).hexdigest()
    signature={'pipeline_revision':IMPLEMENTATION_VERSION,'experiments':[asdict(e) for e in experiments],
               'weights':list(weights),'blend_kind':blend_kind,'folds':folds,'fold_seed':fold_seed,
               'model_seeds':list(model_seeds),'early_stopping':early_stopping,
               'data':{k:fingerprint(v) for k,v in [('train',train),('test',test),('sample',sample)]},
               'versions':{package:version(package) for package in sorted(
                   {'numpy','pandas','scipy','scikit-learn','lightgbm','joblib'} |
                   {e.model.learner for e in experiments if e.model.learner in ('xgboost','catboost')})}}
    signature=json.loads(json.dumps(signature))
    marker=out/'run_signature.json'
    if marker.exists():
        if not resume or json.loads(marker.read_text())!=signature:
            raise ValueError('Existing output has a different run/configuration. Choose a fresh RUN_NAME.')
    elif any(out.iterdir()):
        raise ValueError('Choose an empty output folder or a matching resumable run')
    else:
        marker.write_text(json.dumps(signature,indent=2))
    splits=list(StratifiedKFold(folds,shuffle=True,random_state=fold_seed).split(X,y))
    fold_id=np.empty(len(y),dtype='int16')
    for k,(_,valid) in enumerate(splits):
        fold_id[valid]=k
    pd.DataFrame({ID:train[ID], 'fold':fold_id}).to_csv(out/'fold_assignments.csv',index=False)
    OOF=np.zeros((len(y),len(experiments))); TEST=np.zeros((len(test),len(experiments)))
    rows=[]; started=time.perf_counter()
    transductive_pool=pd.concat([X,XT],ignore_index=True) if any(e.features.frequency_scope=='train_test' for e in experiments) else None
    for j,exp in enumerate(experiments):
        coverage=np.zeros(len(y),dtype='int16')
        for seed in model_seeds:
            for k,(fit,valid) in enumerate(splits):
                directory=out/f'{exp.name}_seed{seed}_fold{k}'
                directory.mkdir(exist_ok=True)
                cache=directory/'complete.npz'
                if resume and cache.exists():
                    with np.load(cache) as data:
                        if not np.array_equal(data['valid_positions'],valid):
                            raise ValueError('Checkpoint fold IDs differ')
                        vp,tp=data['valid'],data['test']
                    row=json.loads((directory/'metrics.json').read_text())
                    print(f'Resumed {exp.name}, seed {seed}, fold {k+1}',flush=True)
                else:
                    start=time.perf_counter()
                    print(f'Fit {exp.name}, seed {seed}, fold {k+1}/{folds}',flush=True)
                    prep=FeaturePipeline(exp.features)
                    a=prep.fit_transform(X.iloc[fit],y[fit],transductive_pool)
                    b=prep.transform(X.iloc[valid]); t=prep.transform(XT)
                    model=ModellingPipeline(replace(exp.model,seed=seed))
                    model.fit(a,y[fit],b if early_stopping else None,y[valid] if early_stopping else None)
                    vp,tp=model.predict(b),model.predict(t)
                    row={'experiment':exp.name,'seed':seed,'fold':k,'auc':float(roc_auc_score(y[valid],vp)),
                         'best_iteration':model.best_iteration_,'features':a.shape[1],
                         'seconds':time.perf_counter()-start}
                    model.save(directory/'model')
                    model.importance().to_csv(directory/'importance.csv',index=False)
                    joblib.dump(prep,directory/'feature_pipeline.joblib',compress=3)
                    (directory/'metrics.json').write_text(json.dumps(row,indent=2))
                    # Write completion marker last; interrupted partial fits are rebuilt.
                    np.savez_compressed(cache,valid_positions=valid,valid=vp,test=tp)
                    del a,b,t,model,prep;gc.collect()
                if len(vp)!=len(valid) or len(tp)!=len(test) or not np.isfinite(vp).all() or not np.isfinite(tp).all():
                    raise ValueError('Invalid checkpoint predictions')
                OOF[valid,j]+=vp/len(model_seeds)
                TEST[:,j]+=tp/(folds*len(model_seeds))
                coverage[valid]+=1
                rows.append(row)
                pd.DataFrame(rows).to_csv(out/'fold_metrics.csv',index=False)
                print(f"  AUC {row['auc']:.7f} | {row['best_iteration']} rounds | {row['seconds']:.1f}s",flush=True)
        if not np.all(coverage==len(model_seeds)):
            raise ValueError('Incomplete OOF coverage')
        save_submission(TEST[:,j],test[ID].to_numpy(),sample,out/f'submission_{exp.name}.csv')
        pd.DataFrame({ID:train[ID],TARGET:y,'fold':fold_id,'prediction':OOF[:,j]}).to_csv(out/f'oof_{exp.name}.csv',index=False)
    names=[e.name for e in experiments]
    oof_blend=blend_predictions(OOF,weights,blend_kind)
    test_blend=blend_predictions(TEST,weights,blend_kind)
    save_submission(test_blend,test[ID].to_numpy(),sample,out/'submission.csv')
    table=pd.DataFrame(OOF,columns=names);table.insert(0,ID,train[ID]);table.insert(1,TARGET,y);table['fold']=fold_id;table['blend']=oof_blend
    table.to_csv(out/'oof_predictions.csv',index=False)
    test_table = pd.DataFrame(TEST, columns=names)
    test_table.insert(0, ID, test[ID].to_numpy())
    test_table['blend'] = test_blend
    test_table.to_csv(out/'test_predictions.csv', index=False)
    pd.DataFrame(OOF,columns=names).corr(method='spearman').to_csv(out/'oof_rank_correlation.csv')
    results=pd.DataFrame([{'experiment':name,'oof_auc':float(roc_auc_score(y,OOF[:,j])),'weight':weights[j]} for j,name in enumerate(names)])
    results.loc[len(results)]=['frozen_blend',float(roc_auc_score(y,oof_blend)),1.0]
    results.to_csv(out/'model_summary.csv',index=False)
    summary={'models':results.to_dict('records'),'elapsed_seconds':time.perf_counter()-started,
             'submission':str((out/'submission.csv').resolve()),'rows':len(test),
             'public_leaderboard_score':None,'uploaded_baseline_lb_user_reported':0.94632,
             'validation_note':'Development OOF. Hyperparameters/features were previously selected using training labels. Public/private LB are unknown until Kaggle scores a submission.'}
    (out/'summary.json').write_text(json.dumps(summary,indent=2))
    print(results.to_string(index=False));print('Submission:',summary['submission'],flush=True)


## 7. Controlled research mode

Feature ablations are kept separate from bounded parameter trials. The research split has distinct fit, stopping, selection, and audit partitions. Early stopping uses the stopping partition, candidate recipes are ranked on selection, and the frozen candidate is evaluated on audit only afterward.

The blend search is intentionally coarse and non-negative. Both probability and logit averaging are evaluated, with at most three members retained. The frozen recipe is written before audit scoring, and the paired bootstrap compares candidate and control predictions on the same audit rows.

The resulting audit report is treated as one development assessment rather than repeated tuning evidence.


In [ ]:
def research_experiments(experiments, train, output_dir, seed=60971, bootstrap=300):
    """Fit/stop/select/audit split: 70/10/10/10, with audit evaluated once.

    A small coarse blend is selected on selection rows. Audit has no role in
    early stopping or weight fitting. Repeated runs on these labels still make
    this a development experiment, not a newly independent test population.
    """
    import itertools
    out=Path(output_dir)
    out.mkdir(parents=True,exist_ok=True)
    if any(out.iterdir()):
        raise ValueError('Use a fresh research output folder for each search')
    y=train[TARGET].map({'No':0,'Yes':1}).to_numpy('int8');X=train[RAW]
    fit,held=train_test_split(np.arange(len(y)),test_size=.3,stratify=y,random_state=seed)
    stop,held=train_test_split(held,test_size=2/3,stratify=y[held],random_state=seed+1)
    select,audit=train_test_split(held,test_size=.5,stratify=y[held],random_state=seed+2)
    arrays=[];audit_arrays=[];rows=[]
    np.savez_compressed(out/'partitions.npz',fit=fit,stop=stop,select=select,audit=audit)
    for exp in experiments:
        if exp.features.frequency_scope!='fold':
            raise ValueError('Research mode uses strict fold-local frequencies')
        start=time.perf_counter();print('Research:',exp.name,flush=True)
        fp=FeaturePipeline(exp.features);a=fp.fit_transform(X.iloc[fit],y[fit]);b=fp.transform(X.iloc[stop])
        model=ModellingPipeline(exp.model).fit(a,y[fit],b,y[stop])
        pred=model.predict(fp.transform(X.iloc[select]))
        arrays.append(pred);audit_arrays.append(model.predict(fp.transform(X.iloc[audit])))
        rows.append({'experiment':exp.name,'selection_auc':float(roc_auc_score(y[select],pred)),
                     'best_iteration':model.best_iteration_,'features':a.shape[1],
                     'seconds':time.perf_counter()-start})
        pd.DataFrame(rows).to_csv(out/'selection_metrics.csv',index=False)
        del fp,a,b,model;gc.collect()
    P=np.column_stack(arrays);A=np.column_stack(audit_arrays)
    # Keep at most three members: include the control and the two strongest
    # challengers. Selection scores are fitting criteria, not confirmation.
    top=[0]+sorted(range(1,len(experiments)),key=lambda j:rows[j]['selection_auc'],reverse=True)[:2]
    trials=[]
    for kind in ['probability','logit']:
        for ticks in itertools.product(range(5),repeat=len(top)):
            if sum(ticks)!=4:
                continue
            weight=np.asarray(ticks)/4
            score=roc_auc_score(y[select],blend_predictions(P[:,top],weight,kind))
            trials.append({'kind':kind,'weights':weight.tolist(),'selection_auc':float(score)})
    best=max(trials,key=lambda t:t['selection_auc'])
    chosen=[j for j,w in zip(top,best['weights']) if w>0]
    weights=[w for w in best['weights'] if w>0]
    selected_specs=[asdict(experiments[j]) for j in chosen]
    for spec,j in zip(selected_specs,chosen):
        spec['model']['rounds']=rows[j]['best_iteration']
    frozen={'experiments':selected_specs,'weights':weights,'kind':best['kind'],
            'selection_auc':best['selection_auc'],'iterations':[rows[j]['best_iteration'] for j in chosen]}
    (out/'frozen_recipe.json').write_text(json.dumps(frozen,indent=2))
    audit_pred=blend_predictions(A[:,chosen],weights,best['kind'])
    baseline=float(roc_auc_score(y[audit],A[:,0]));score=float(roc_auc_score(y[audit],audit_pred))
    interval=paired_auc_interval(y[audit],A[:,0],audit_pred,seed+100,bootstrap)
    report={'control_audit_auc':baseline,'candidate_audit_auc':score,'delta':score-baseline,
            'paired_95pct_interval':interval,'positive_confirmation':bool(interval[0]>0),
            'public_leaderboard_score':None,
            'note':'Recipe frozen before audit. Audit comparison is one assessment; do not repeatedly tune against it.'}
    (out/'audit_report.json').write_text(json.dumps(report,indent=2))
    pd.DataFrame(A,columns=[e.name for e in experiments]).assign(target=y[audit]).to_csv(out/'audit_predictions.csv',index=False)
    print(pd.DataFrame(rows).sort_values('selection_auc',ascending=False).to_string(index=False))
    print(json.dumps(report,indent=2))
    return frozen,report


def conditional_lgb_trials(base, count=12, seed=609, prefix='tune'):
    """Reproducible bounded random search; no Optuna dependency or inactive dimensions."""
    rng=np.random.default_rng(seed);result=[];seen=set()
    for attempt in range(max(100,count*100)):
        if len(result)==count:
            break
        depth=int(rng.choice([4,5,6]))
        leaves=int(rng.choice([n for n in [15,23,31,47,63] if n<=2**depth]))
        bag=float(rng.choice([1.,.85,.95]))
        params=dict(max_depth=depth,num_leaves=leaves,min_data_in_leaf=int(rng.choice([10,20,50])),
                    feature_fraction=float(rng.choice([.3,.45,.6])),bagging_fraction=bag,
                    bagging_freq=0 if bag==1 else 1,lambda_l1=float(rng.choice([0,.071,.2])),
                    lambda_l2=float(rng.choice([2.,5.,10.])),max_bin=int(rng.choice([1024,4095])),
                    learning_rate=float(rng.choice([.015,.02,.03])))
        key=json.dumps(params,sort_keys=True)
        if key not in seen:
            seen.add(key);result.append(make_experiment(f'{prefix}_{len(result):02d}',base,params=params))
    return result

CONTROL = Experiment("dual_control", FeatureConfig(), ModelConfig(threads=THREADS))
RESEARCH_CANDIDATES = [
    CONTROL,
    make_experiment("income_modulo", CONTROL, features={"income_moduli": (100,1000)}),
    make_experiment("income_neighborhood", CONTROL,
                    features={"income_bins": (8192,16384), "neighborhood_smoothing": 1.}),
    make_experiment("wide_bins", CONTROL, params={"max_bin": 16383}),
    make_experiment("active_bagging", CONTROL,
                    params={"bagging_fraction": .85, "bagging_freq": 1}),
]
if EXTRA_TUNING_TRIALS:
    RESEARCH_CANDIDATES += conditional_lgb_trials(
        RESEARCH_CANDIDATES[2], count=EXTRA_TUNING_TRIALS, seed=609)
if SMOKE_TEST:
    for exp in RESEARCH_CANDIDATES:
        exp.model.rounds, exp.model.patience = 20, 5
        exp.features.inner_folds = 3


## 8. Historical research archive

The following recorded experiments predate this revision. They explain the original model choice; their scores are not measurements of today's smooth-key/triple-TE recipe.


In [ ]:
LOCAL_RESEARCH = {'experiments': [{'name': 'smooth1_bagging',
                  'select_auc': 0.9456525378975227,
                  'stop_auc': 0.9446378467893575,
                  'best_iteration': 1377,
                  'features': 90,
                  'seconds': 45.889951900113374,
                  'round_ceiling_hit': False},
                 {'name': 'xgb_smooth1',
                  'select_auc': 0.9456039302343409,
                  'stop_auc': 0.944699763624149,
                  'best_iteration': 1497,
                  'features': 90,
                  'seconds': 70.28755340003408,
                  'round_ceiling_hit': False},
                 {'name': 'neigh_kernel',
                  'select_auc': 0.9456026214620514,
                  'stop_auc': 0.9446241412824166,
                  'best_iteration': 1286,
                  'features': 91,
                  'seconds': 48.97412999998778,
                  'round_ceiling_hit': False},
                 {'name': 'catboost_neighborhood',
                  'select_auc': 0.9455898052686789,
                  'stop_auc': 0.9445752356022105,
                  'best_iteration': 944,
                  'features': 90,
                  'seconds': 73.53685849998146,
                  'round_ceiling_hit': False},
                 {'name': 'neigh_smooth1',
                  'select_auc': 0.9455780488162351,
                  'stop_auc': 0.944584460015067,
                  'best_iteration': 1146,
                  'features': 90,
                  'seconds': 43.56607489986345,
                  'round_ceiling_hit': False},
                 {'name': 'neigh_depth4',
                  'select_auc': 0.9455639913450539,
                  'stop_auc': 0.9446887385500443,
                  'best_iteration': 2081,
                  'features': 90,
                  'seconds': 53.548240300035104,
                  'round_ceiling_hit': False},
                 {'name': 'smooth1_depth4',
                  'select_auc': 0.945559181393546,
                  'stop_auc': 0.944664460267796,
                  'best_iteration': 1978,
                  'features': 90,
                  'seconds': 51.92893349984661,
                  'round_ceiling_hit': False},
                 {'name': 'neigh_bagging',
                  'select_auc': 0.94555748549774,
                  'stop_auc': 0.9446557354783104,
                  'best_iteration': 1262,
                  'features': 90,
                  'seconds': 47.284648600034416,
                  'round_ceiling_hit': False},
                 {'name': 'neighborhood_v2',
                  'select_auc': 0.9455316560581427,
                  'stop_auc': 0.9447092727602618,
                  'best_iteration': 1719,
                  'features': 90,
                  'seconds': 57.436530200066045,
                  'round_ceiling_hit': False},
                 {'name': 'xgb_neighborhood',
                  'select_auc': 0.9455236187843327,
                  'stop_auc': 0.9447297425780266,
                  'best_iteration': 1565,
                  'features': 90,
                  'seconds': 98.75765819987282,
                  'round_ceiling_hit': False},
                 {'name': 'dual_neighborhood',
                  'select_auc': 0.9455102145355985,
                  'stop_auc': 0.9446532482228334,
                  'best_iteration': 1644,
                  'features': 90,
                  'seconds': 59.74132230016403,
                  'round_ceiling_hit': False},
                 {'name': 'neigh_l2_10',
                  'select_auc': 0.9455003603413963,
                  'stop_auc': 0.9446945959358364,
                  'best_iteration': 1782,
                  'features': 90,
                  'seconds': 55.95130189997144,
                  'round_ceiling_hit': False},
                 {'name': 'neigh_float_digits',
                  'select_auc': 0.9454946566698663,
                  'stop_auc': 0.944695135901467,
                  'best_iteration': 2036,
                  'features': 254,
                  'seconds': 152.67243420006707,
                  'round_ceiling_hit': False},
                 {'name': 'neigh_interactions',
                  'select_auc': 0.9454854712140833,
                  'stop_auc': 0.9445790805298909,
                  'best_iteration': 1346,
                  'features': 101,
                  'seconds': 51.42594389989972,
                  'round_ceiling_hit': False},
                 {'name': 'neigh_col60',
                  'select_auc': 0.9454724657258449,
                  'stop_auc': 0.9446516624617,
                  'best_iteration': 1446,
                  'features': 90,
                  'seconds': 57.07507030013949,
                  'round_ceiling_hit': False},
                 {'name': 'neigh_depth6',
                  'select_auc': 0.9454663508810409,
                  'stop_auc': 0.9446280886173718,
                  'best_iteration': 1394,
                  'features': 90,
                  'seconds': 53.59781279996969,
                  'round_ceiling_hit': False},
                 {'name': 'neigh_modulo',
                  'select_auc': 0.9454589202817598,
                  'stop_auc': 0.9447304462401344,
                  'best_iteration': 2038,
                  'features': 98,
                  'seconds': 64.95646360004321,
                  'round_ceiling_hit': False},
                 {'name': 'dual_modulo',
                  'select_auc': 0.9454248642734865,
                  'stop_auc': 0.9445795506723794,
                  'best_iteration': 1520,
                  'features': 92,
                  'seconds': 68.37528180005029,
                  'round_ceiling_hit': False},
                 {'name': 'dual_16383',
                  'select_auc': 0.9453919859275343,
                  'stop_auc': 0.9445432038479615,
                  'best_iteration': 1650,
                  'features': 80,
                  'seconds': 65.0343688998837,
                  'round_ceiling_hit': False},
                 {'name': 'dual_control',
                  'select_auc': 0.9453856926490449,
                  'stop_auc': 0.944539960950927,
                  'best_iteration': 1757,
                  'features': 80,
                  'seconds': 65.14085900015198,
                  'round_ceiling_hit': False},
                 {'name': 'linear_nested',
                  'select_auc': 0.9453700060007403,
                  'stop_auc': 0.9440473540300048,
                  'best_iteration': 967,
                  'features': 21,
                  'seconds': 26.68983000004664,
                  'round_ceiling_hit': False},
                 {'name': 'linear_nested_unregularized',
                  'select_auc': 0.9452722724408916,
                  'stop_auc': 0.9438637300282074,
                  'best_iteration': 480,
                  'features': 21,
                  'seconds': 14.008659600047395,
                  'round_ceiling_hit': False},
                 {'name': 'legacy_anchor_reference',
                  'select_auc': 0.9451911083881889,
                  'stop_auc': 0.9440524681872422,
                  'best_iteration': 782,
                  'features': 21,
                  'seconds': 15.075903000077233,
                  'round_ceiling_hit': False,
                  'reference_only': True,
                  'note': 'Existing self-including training TE anchor; outer stop/select/audit '
                          'labels excluded from feature fitting. Not a default final member.'},
                 {'name': 'raw_16383',
                  'select_auc': 0.9438549721295738,
                  'stop_auc': 0.9426272335600729,
                  'best_iteration': 4233,
                  'features': 13,
                  'seconds': 60.07525520003401,
                  'round_ceiling_hit': False},
                 {'name': 'raw_fine63',
                  'select_auc': 0.9435273415068146,
                  'stop_auc': 0.9423442760533696,
                  'best_iteration': 777,
                  'features': 13,
                  'seconds': 11.372296900022775,
                  'round_ceiling_hit': False},
                 {'name': 'raw_255',
                  'select_auc': 0.9418551075591891,
                  'stop_auc': 0.9406044323134649,
                  'best_iteration': 1692,
                  'features': 13,
                  'seconds': 25.125311099924147,
                  'round_ceiling_hit': False}],
 'audit': {'control': 'dual_control',
           'control_audit_auc': 0.9450166157602603,
           'candidate_audit_auc': 0.94516781736171,
           'delta': 0.00015120160144965755,
           'paired_95pct_interval': [-3.168749505486623e-05, 0.00034783191482333774],
           'positive_confirmation': False,
           'selection_auc': 0.9457758138514757,
           'public_leaderboard_score': None,
           'public_target_snapshot': 0.94672,
           'note': 'One frozen-recipe assessment after selection; previously used training '
                   'population, not a completely untouched dataset. AUC cannot be compared '
                   'directly to public LB.',
           'legacy_anchor_audit_auc': 0.9444331360588097,
           'delta_vs_legacy_anchor': 0.0007346813029002197,
           'paired_interval_vs_legacy': [0.0005302539417622431, 0.0009499712800899062]},
 'fit_rows': 468065,
 'stop_rows': 66866,
 'selection_rows': 66867,
 'audit_rows': 66867,
 'data_hashes': {'train.csv': 'eae9eaa4e6378df405e755f853771d7e26d212bd93258349fc797b771021946a',
                 'test.csv': '539263f6caabc40afd5e2f0bc0ab16b10a2d1177c565fc71b866f0181d836b34'},
 'selected_names': ['smooth1_bagging', 'catboost_neighborhood', 'linear_nested'],
 'blend_trials': 130}
display(pd.DataFrame(LOCAL_RESEARCH["experiments"]).sort_values("select_auc", ascending=False))
display(pd.DataFrame([LOCAL_RESEARCH["audit"]]))


### Insights from 26 controlled configurations

Across **26 configurations**, including the previous compact anchor as a reference, all candidates were compared on the same fit/stopping/selection partitions.

- **Representation mattered more than raw histogram resolution.** The raw 255-bin LightGBM reached **0.941855** selection AUC. Raising raw resolution to 16,383 bins improved it to **0.943855**, while the dual-encoding control reached **0.945386**. Once exact-value encodings were present, simply increasing `max_bin` added little.
- **Income neighborhoods produced the strongest selected single model.** `smooth1_bagging` reached **0.945653** selection AUC with weak neighborhood smoothing and active bagging.
- **A Gaussian neighborhood was competitive.** The `neigh_kernel` variant reached **0.945603**, suggesting that smooth local income structure and multiscale bins capture related but not identical signal.
- **More engineered columns were not automatically better.** The floating-point digit representation used **254 features**, took about **2.7×** the runtime of the neighborhood reference, and scored lower. The tested domain-interaction block also failed to improve the neighborhood reference.
- **The final blend won selection but did not prove a clean audit gain.** The frozen blend reached **0.945776** on selection and **0.945168** on audit. Its audit delta over `dual_control` was **+0.000151**, with a paired 95% bootstrap interval of **[-0.000032, +0.000348]**. Because the interval includes zero, this audit alone does not support a reliable improvement claim over the control.
- **Implementation provenance matters.** Two early neighborhood trials used an earlier bin-edge implementation. They remain in the research history but were excluded from the final blend search. The selected neighborhood members use shared outer-training bin edges.

> **Main takeaway:** exact-value and local-income representations produced the clearest improvement over raw numeric modelling in this experiment series, while the ensemble gain was small enough that its uncertainty deserves equal visibility with the headline score.


## 9. Train and save component predictions

The default makes 15 fits: three models, five folds, one seed. Every outer fold fits its own feature pipeline. Matching checkpoints can resume; changed settings require a fresh `RUN_NAME`.

Research mode retains the historical search separately. Use submission mode for this improved recipe.


In [ ]:
output_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
output_dir = output_root / (RUN_NAME + ("_smoke" if SMOKE_TEST else ""))
if RUN_MODE == "submission":
    result = run_cross_validation(
        EXPERIMENTS, train, test, sample_submission, BLEND_WEIGHTS, output_dir,
        folds=FOLDS, fold_seed=FOLD_SEED, model_seeds=MODEL_SEEDS,
        blend_kind=BLEND_KIND, early_stopping=EARLY_STOP_ON_OUTER_FOLD, resume=True)
elif RUN_MODE == "research":
    frozen_research, research_audit = research_experiments(
        RESEARCH_CANDIDATES, train, output_dir, seed=60971,
        bootstrap=30 if SMOKE_TEST else 300)
else:
    raise ValueError("RUN_MODE must be submission or research")


## 10. OOF diagnostics and model stability

Models are compared on the same development OOF population, while the public leaderboard snapshot stays off the OOF axis because the samples are different. The diagnostics include model-level AUC, fold-level AUC, runtime summaries, and Spearman rank correlation across component predictions.

The fold plot is a stability diagnostic rather than a leaderboard proxy. A blend member can contribute through complementary errors even when it is not the best standalone model, so both score and correlation matter.


In [ ]:
import matplotlib.pyplot as plt

if RUN_MODE == "submission":
    metrics = pd.read_csv(output_dir / "model_summary.csv")
    fold_metrics = pd.read_csv(output_dir / "fold_metrics.csv")
    display(metrics)
    display(fold_metrics.groupby("experiment")[["auc", "seconds"]].agg(["mean", "std"]))
    display(pd.read_csv(output_dir / "oof_rank_correlation.csv", index_col=0))
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    ordered = metrics.sort_values("oof_auc")
    axes[0].barh(ordered.experiment, ordered.oof_auc, color="#287d8e")
    lo, hi = ordered.oof_auc.min(), ordered.oof_auc.max()
    axes[0].set_xlim(max(0, lo-.001), min(1, hi+.0003))
    axes[0].set_title("Development OOF AUC")
    for i, value in enumerate(ordered.oof_auc):
        axes[0].text(value, i, f" {value:.6f}", va="center", fontsize=9)
    for name, group in fold_metrics.groupby("experiment"):
        axes[1].plot(group.groupby("fold").auc.mean().index+1,
                     group.groupby("fold").auc.mean().values, marker="o", label=name)
    axes[1].set_xlabel("Validation fold")
    axes[1].set_xticks(sorted(fold_metrics["fold"].unique()+1))
    axes[1].set_ylabel("AUC")
    axes[1].set_title("Fold stability")
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(output_dir / "model_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()


### Feature importance across folds

Feature importance is normalized within each fitted fold and averaged by learner. These tables describe how the trained models used the available features; they do not establish causal importance. Correlated raw, digit, target-encoding, and neighborhood columns can substitute for one another, and LightGBM gain importance is not numerically equivalent to CatBoost importance.


In [ ]:
if RUN_MODE == "submission":
    for exp in EXPERIMENTS:
        blocks = []
        for path in sorted(output_dir.glob(f"{exp.name}_seed*_fold*/importance.csv")):
            frame = pd.read_csv(path)
            total = frame["importance"].sum()
            frame["share"] = frame["importance"] / total if total > 0 else 0.0
            blocks.append(frame[["feature", "share"]])
        importance = (pd.concat(blocks).groupby("feature")["share"].mean()
                      .sort_values(ascending=False).rename("mean_importance_share"))
        importance.to_csv(output_dir / f"importance_{exp.name}.csv")
        print(exp.name)
        display(importance.head(12).to_frame())


## 11. Submission and saved artifacts

Before the root Kaggle file is written, the final submission is checked for row count, ID order, and valid probability range. The run also retains component submissions, OOF predictions, fold assignments, fold metrics, importance tables, rank correlations, fitted preprocessors, model files, and the run signature.

This artifact trail keeps the final blend reproducible and preserves component-level comparisons without obscuring which member produced each prediction set.


In [ ]:
if RUN_MODE == "submission":
    submission = pd.read_csv(output_dir / "submission.csv")
    assert len(submission) == len(test)
    assert submission[ID].tolist() == sample_submission[ID].tolist()
    assert submission[ID].is_unique
    assert np.isfinite(submission[TARGET]).all()
    assert submission[TARGET].between(0, 1).all()
    if not SMOKE_TEST:
        submission.to_csv(output_root / "submission.csv", index=False)
        print("READY:", (output_root / "submission.csv").resolve())
    else:
        print("Smoke checks passed; no root competition submission was written.")
    display(submission.head())
    print("Artifacts:", output_dir.resolve())
    print("Uploaded baseline LB: 0.94632 (user-reported). This revision: not yet scored.")


## What to compare after running

1. Submit the root `submission.csv` and compare its actual LB with the uploaded baseline **0.94632**.
2. Keep the full output directory: aligned OOF/test matrices allow future model comparisons without retraining.
3. Check fold stability and individual model scores. A higher development OOF score does not guarantee a higher LB.
4. Keep the separate **0.94650 public-output submission** as your best scored reference. This notebook does not import it.

The historical research tables above are retained for attribution and context. Their old scores and leaderboard snapshots are not results of this revision.

## Sources and citations

**Competition and API references**

1. [Kaggle — Playground Series S6E9 leaderboard](https://www.kaggle.com/competitions/playground-series-s6e9/leaderboard)
2. [scikit-learn — TargetEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.TargetEncoder.html)
14. [LightGBM — Parameters](https://lightgbm.readthedocs.io/en/stable/Parameters.html)
15. [LightGBM — Parameters Tuning](https://lightgbm.readthedocs.io/en/stable/Parameters-Tuning.html)

**Public notebook research references**

3. [Georgy Mamarin — fixed comparisons, noise controls and bin-resolution experiments](https://www.kaggle.com/code/georgymamarin/s6e9-starter-how-to-tell-a-real-gain-from-noise)
4. [Naji — shallow LightGBM, digits and dual target encoding](https://www.kaggle.com/code/najiama/pure-lgbm-model-cv-0-94587-lb-0-94612)
5. [Kirill — LightGBM representation experiments](https://www.kaggle.com/code/kirill0212/s6e9-lightgbm)
6. [Mizushima — explicit nested target and frequency encoding](https://www.kaggle.com/code/mizushimatoshihiko/s6e9-leak-free-target-frequency-encoding-lgbm)
7. [jazivxt — multiscale income-neighborhood channels](https://www.kaggle.com/code/jazivxt/single-model-zoom-zoom)
8. [megayak — income modulo diagnostics](https://www.kaggle.com/code/megayak/s6e9-the-1000-leak)
9. [Sergey — compact linear-leaf model and Gaussian income idea](https://www.kaggle.com/code/sergeyqt2024/simple-21-feature-lfbm-0-9456)
10. [Koushik — charging and policy interactions](https://www.kaggle.com/code/koushikkumardinda/s6e9-feature-eng-rank-ensemble-baseline)
11. [Rugved — multiple target-encoding smoothing channels](https://www.kaggle.com/code/rugvedbane/lightgbm-triple-target-encoding-0-94590-lb)
12. [Chris Deotte — original-data insights](https://www.kaggle.com/code/cdeotte/fable-5-1-eda-original-data-insights)
13. [Yusuke Hayashi — original-dataset reconstruction](https://www.kaggle.com/code/yhay81/exact-reconstruction-of-the-ev-source-dataset)
